# MOT — eksploracja i wizualizacja (`./data`)

**Uwaga:** animacje `to_jshtml()` robią bardzo duży HTML (base64 PNG). Żeby nie uszkodzić `.ipynb` przy zapisie — kończąc pracę wyczyść outputy (*Edit → Clear All Outputs*) albo nie zapisuj notebooka po uruchomieniu ciężkich komórek.

Trening: nakładka `gt/gt.txt` (`eval_flag==1`, `cls==1`). Test: tylko `det/det.txt` (bez GT).


In [ ]:
# Uruchom z korzenia repozytorium lub notebooks/. Zależności: pip install -r requirements.txt (folder wyzej)
from __future__ import annotations

import configparser
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt

plt.rcParams["animation.embed_limit"] = 128  # MB — domyślnie ~20 MB; bez tego/innych limitów notebook puchnie lub się psuje przy zapisie
import numpy as np
from IPython.display import HTML, Markdown, display
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Rectangle


def resolve_data_root() -> Path:
    for rel in (Path("data"), Path("../data"), Path("../../data")):
        p = (Path.cwd() / rel).resolve()
        if (p / "evs_mot-train").is_dir() and (p / "evs_mot-test").is_dir():
            return p
    raise FileNotFoundError(
        "Brak folderu data z evs_mot-train i evs_mot-test. Uruchom Jupyter z korzenia repozytorium lub ustaw cwd."
    )


DATA_ROOT = resolve_data_root()
TRAIN_SPLIT = DATA_ROOT / "evs_mot-train"
TEST_SPLIT = DATA_ROOT / "evs_mot-test"

FRAME_STRIDE = 4
MAX_FRAMES = 40
ANIM_INTERVAL_MS = 65

In [ ]:
@dataclass
class DetRow:
    frame: int
    x: float
    y: float
    w: float
    h: float
    conf: float


@dataclass
class GtRow:
    frame: int
    track_id: int
    x: float
    y: float
    w: float
    h: float
    eval_flag: int
    cls: int


def parse_seqinfo(seq_dir: Path) -> dict:
    cfg = configparser.ConfigParser()
    cfg.read(seq_dir / "seqinfo.ini", encoding="utf-8")
    s = cfg["Sequence"]
    return {
        "seqLength": int(s["seqLength"]),
        "imDir": s["imDir"],
        "imExt": s.get("imExt", ".jpg").strip() or ".jpg",
    }


def iter_sequence_dirs(split_root: Path) -> list[Path]:
    if not split_root.is_dir():
        return []
    return sorted(p for p in split_root.iterdir() if p.is_dir())


def find_image_path(seq_dir: Path, frame: int, info: dict) -> Path | None:
    im_dir = seq_dir / info["imDir"]
    ext = info["imExt"]
    if not str(ext).startswith("."):
        ext = "." + ext
    direct = im_dir / f"{frame:06d}{ext}"
    if direct.exists():
        return direct
    for e in (".jpg", ".jpeg", ".png", ".bmp"):
        alt = im_dir / f"{frame:06d}{e}"
        if alt.exists():
            return alt
    return None


def load_det(path: Path) -> list[DetRow]:
    rows: list[DetRow] = []
    if not path.exists():
        return rows
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line:
            continue
        frame, _, x, y, w, h, conf = line.split(",")
        rows.append(DetRow(int(frame), float(x), float(y), float(w), float(h), float(conf)))
    return rows


def load_gt(path: Path) -> list[GtRow]:
    rows: list[GtRow] = []
    if not path.exists():
        return rows
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line:
            continue
        frame, tid, x, y, w, h, ef, cls, _vis = line.split(",")
        rows.append(
            GtRow(
                int(frame),
                int(tid),
                float(x),
                float(y),
                float(w),
                float(h),
                int(ef),
                int(cls),
            )
        )
    return rows


def color_for_track(track_id: int) -> tuple:
    cmap = plt.get_cmap("tab20")
    return cmap(track_id % 20)[:3]


def build_frame_list(seq_len: int, stride: int, max_frames: int) -> list[int]:
    frames = list(range(1, seq_len + 1, stride))
    if len(frames) <= max_frames:
        return frames
    idx = np.linspace(0, len(frames) - 1, max_frames, dtype=int)
    return [frames[int(i)] for i in idx]


def summarize_split(split_root: Path, label: str) -> None:
    print(f"=== {label} ===")
    for seq_dir in iter_sequence_dirs(split_root):
        det_p = seq_dir / "det" / "det.txt"
        gt_p = seq_dir / "gt" / "gt.txt"
        info = parse_seqinfo(seq_dir)
        dets = load_det(det_p)
        gts = load_gt(gt_p)
        gt_eval = [g for g in gts if g.eval_flag == 1 and g.cls == 1]
        d_frames = [d.frame for d in dets]
        g_frames = [g.frame for g in gt_eval]
        print(
            f"  {seq_dir.name}: len={info['seqLength']} | det={len(dets)} | "
            f"gt_eval_people={len(gt_eval)} | det_frames [{min(d_frames, default=0)}..{max(d_frames, default=0)}] "
            f"| gt_frames [{min(g_frames, default=0)}..{max(g_frames, default=0)}]"
        )


summarize_split(TRAIN_SPLIT, "train")
summarize_split(TEST_SPLIT, "test")


In [ ]:
def animate_sequence(seq_dir: Path, box_mode: str) -> HTML:
    if box_mode not in {"none", "gt", "det"}:
        raise ValueError(box_mode)
    info = parse_seqinfo(seq_dir)
    seq_len = info["seqLength"]
    frames = build_frame_list(seq_len, FRAME_STRIDE, MAX_FRAMES)
    if not frames:
        return HTML("<p>Brak klatek</p>")

    gt_by_f: dict[int, list[GtRow]] = defaultdict(list)
    det_by_f: dict[int, list[DetRow]] = defaultdict(list)
    if box_mode == "gt":
        for g in load_gt(seq_dir / "gt" / "gt.txt"):
            if g.eval_flag == 1 and g.cls == 1:
                gt_by_f[g.frame].append(g)
    elif box_mode == "det":
        for d in load_det(seq_dir / "det" / "det.txt"):
            det_by_f[d.frame].append(d)

    fig, ax = plt.subplots(figsize=(7, 4), dpi=72)

    def draw_frame(i: int):
        ax.clear()
        fnum = frames[i]
        img_path = find_image_path(seq_dir, fnum, info)
        title = f"{seq_dir.name} | frame {fnum}/{seq_len} | {box_mode}"
        if img_path is None:
            ax.set_title(title + " | BRAK OBRAZU")
            ax.axis("off")
            return
        ax.imshow(plt.imread(img_path))
        if box_mode == "gt":
            for g in gt_by_f.get(fnum, []):
                c = color_for_track(g.track_id)
                ax.add_patch(
                    Rectangle((g.x, g.y), g.w, g.h, fill=False, edgecolor=c, linewidth=2.0)
                )
                ax.text(
                    g.x,
                    max(0, g.y - 4),
                    str(g.track_id),
                    color=c,
                    fontsize=8,
                    bbox={"facecolor": "black", "alpha": 0.35, "pad": 1, "edgecolor": "none"},
                )
        elif box_mode == "det":
            for d in det_by_f.get(fnum, []):
                ax.add_patch(
                    Rectangle(
                        (d.x, d.y),
                        d.w,
                        d.h,
                        fill=False,
                        edgecolor="darkorange",
                        linewidth=1.6,
                        linestyle="--",
                    )
                )
        ax.set_title(title)
        ax.axis("off")

    anim = FuncAnimation(
        fig,
        draw_frame,
        frames=len(frames),
        interval=ANIM_INTERVAL_MS,
        repeat=True,
    )
    html = anim.to_jshtml()
    plt.close(fig)
    return HTML(html)


def show_sequence_pair(seq_dir: Path, train: bool) -> None:
    display(Markdown(f"### {seq_dir.name}"))
    if train:
        display(animate_sequence(seq_dir, "gt"))
    else:
        display(animate_sequence(seq_dir, "det"))

## Zbiór treningowy (`evs_mot-train`)

In [ ]:
for seq_dir in iter_sequence_dirs(TRAIN_SPLIT):
    show_sequence_pair(seq_dir, train=True)

## Zbiór testowy (`evs_mot-test`)

In [ ]:
for seq_dir in iter_sequence_dirs(TEST_SPLIT):
    show_sequence_pair(seq_dir, train=False)